# 12 - Tổng hợp kết quả nghiên cứu chỉ đọc

**Mục tiêu:** Đủ 12 phần từ manifest cụ thể; truy nguồn bảng/hình/run, giải thích và giới hạn; không tạo kết quả mới.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.

**Chạy nhóm:** chủ thư mục chia sẻ `PUBG_Project` với quyền Editor. Mỗi thành viên thêm shortcut của chính thư mục đó vào My Drive, chọn `drive` và bật `PUBG_REQUIRE_EXISTING_PROJECT = True`. Mỗi người mount Drive của mình; kết quả phải nằm trong cùng thư mục gốc được chia sẻ. Chạy xong notebook, chờ file hiện trên Drive rồi bàn giao cho người tiếp theo; mỗi lần chỉ một người ghi. Người nhận chạy cell cấu hình, Bootstrap và khởi tạo của notebook tiếp theo. Biến trong RAM không được chuyển sang phiên mới; cell đang chạy dở có thể phải chạy lại. Xem `TEAM_DRIVE.md` để thiết lập và xác nhận đường dẫn.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "drive"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/PUBG_Project/Project_PUBG"  # @param {type:"string"}
# @markdown Nhóm dùng cùng thư mục đã chia sẻ: bật True để tránh tạo nhầm project riêng khi thiếu shortcut.
PUBG_REQUIRE_EXISTING_PROJECT = True  # @param {type:"boolean"}
# @markdown Số dòng mỗi batch khi đọc CSV trong ZIP; giảm nếu RAM ít. Không lấy mẫu dữ liệu.
PUBG_BATCH_ROWS = 50000  # @param {type:"integer"}
# Optional team label. Blank uses a unique runtime process ID.
PUBG_WRITER_ID = ""  # @param {type:"string"}
# Set True only after confirming the previous writer runtime has stopped.
PUBG_FORCE_STAGE_TAKEOVER = False  # @param {type:"boolean"}

# @markdown Chọn đúng manifest snapshot hoặc canonical cụ thể; không tìm latest.
PUBG_SUMMARY_MANIFEST = ""  # @param {type:"string"}
# Chỉ bật cho fixture, không dùng làm approval kết quả nghiên cứu.
PUBG_SUMMARY_ALLOW_FIXTURE = False  # @param {type:"boolean"}


In [ ]:
# Bootstrap chỉ đọc: không giải nén bundle, tạo thư mục, cài package hoặc ghi checkpoint.
import os
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "drive").strip().lower()
if PUBG_STORAGE_MODE not in {"drive", "runtime"}:
    raise ValueError("PUBG_STORAGE_MODE phải là drive hoặc runtime")
if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Máy local chọn runtime và đường dẫn manifest cụ thể; Drive mount chỉ trên Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(PUBG_DRIVE_PROJECT_ROOT).expanduser().resolve()
    if PUBG_REQUIRE_EXISTING_PROJECT and not PROJECT_ROOT.is_dir():
        raise FileNotFoundError("Không thấy project đã chia sẻ; kiểm tra shortcut/quyền. Không tạo project mới.")
else:
    PROJECT_ROOT = Path(globals().get("PUBG_SUMMARY_CODE_ROOT") or Path.cwd()).expanduser().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
_selected_manifest = globals().get("PUBG_SUMMARY_MANIFEST", "")
if not str(_selected_manifest).strip():
    raise ValueError("Điền PUBG_SUMMARY_MANIFEST bằng đường dẫn một release cụ thể; không tìm latest.")
manifest_path = Path(_selected_manifest).expanduser()
if not manifest_path.is_absolute():
    manifest_path = PROJECT_ROOT / manifest_path
manifest_path = manifest_path.resolve()
if not manifest_path.is_file():
    raise FileNotFoundError("Không thấy manifest đã chọn: " + str(manifest_path))
try:
    from src.evaluation.finalize import load_locked_release
    from src.evaluation.summary import render_summary_section
except ImportError as error:
    raise ImportError("Cần source hiện hành và dependencies đọc báo cáo; đồng bộ src hoặc cài requirements ngoài summary. Không tự sửa release.") from error
PUBG_SUMMARY_ALLOW_FIXTURE = globals().get("PUBG_SUMMARY_ALLOW_FIXTURE", False)
_PUBG_CELL_PROGRESS = {}
print("Chế độ chỉ đọc; manifest đã chọn:", manifest_path)


### 0. Bối cảnh, mục tiêu và kiểm soát chỉ đọc

Notebook này nối RQ-phương pháp-kết quả-hạn chế để nhóm viết báo cáo từ đúng một release đã khóa. Nó không chạy raw, download, build, train, chọn model hoặc sinh metric/hình mới. Đầu vào là `PUBG_SUMMARY_MANIFEST`; điền path snapshot cụ thể hoặc canonical cụ thể. Không scan latest.

Bootstrap chỉ mount Drive nếu cần và chuẩn bị imports trong RAM; không tạo thư mục, giải nén bundle, sửa config, cài packages hoặc ghi checkpoint. Cần đồng bộ source hiện hành/dependencies từ trước. Không cần raw/processed hoặc RAM phiên trước. Một notebook khác đang running/stale không chặn snapshot hợp lệ.

Hai kiểm tra khác nhau: integrity đối chiếu bytes/SHA/schema từng file; completeness đối chiếu required tasks/selected executions và conditional exceptions. Legacy integrity inventory không phải release hoàn chỉnh. Mọi lỗi phải dừng trước báo cáo, không sửa hash để ép pass.

Fixture chỉ được mở khi `PUBG_SUMMARY_ALLOW_FIXTURE=True`; hiển thị nhãn kiểm thử và không biến hình fixture thành report-ready. N artifact khác N dòng/người/trận. Bảng xem giới hạn100dòng có disclosure, artifact lưu vẫn đầy đủ; không công bố player identifiers không cần thiết.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
_pubg_progress['12_final_results_summary.ipynb'] = -1
import pandas as pd
from IPython.display import display, Markdown
from src.evaluation.finalize import load_locked_release
from src.evaluation.summary import render_summary_section
release = load_locked_release(manifest_path, allow_fixture=PUBG_SUMMARY_ALLOW_FIXTURE)
display(Markdown("Kiểm thử logic, không phải kết quả PUBG chính thức." if release['fixture'] else "Đọc release report-ready đã khóa; không tạo run mới."))
display(pd.DataFrame(release['audit']))
display(pd.DataFrame(release['manifest']['required_matrix']))
display(pd.DataFrame([{'Thí nghiệm': key, 'Run đã xác minh': value} for key,value in release['manifest']['official_run_ids'].items()]))
summary_views = []
_pubg_figure_details = {}

_pubg_progress['12_final_results_summary.ipynb'] = 5


### 1. Tóm tắt dữ liệu

Câu hỏi: Release phân tích nguồn và population nào?

Input/phương pháp: Đọc source inventory, cohort và cohort ledger đã khóa. N dòng, trận, người chơi khác nhau; không suy số người từ số dòng.

Cách đọc: Đối chiếu source/version, mode, khoảng thời gian và counts theo split; unknown/null là chưa có evidence, không bằng 0.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Full nghĩa toàn cohort hợp lệ của từng task; nguồn cũ hoặc scope fixture không đại diện toàn PUBG.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 5:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 5
summary_views.append(render_summary_section(release, 1))

_pubg_progress['12_final_results_summary.ipynb'] = 7


### 2. Chất lượng dữ liệu

Câu hỏi: Missing, exclusions, join/roster và chronology có evidence nào?

Input/phương pháp: Chỉ đọc audit selected. Flags có thể chồng lặp; removal cascade mới là số loại tuần tự. Structural missing khác parse/error/unavailable.

Cách đọc: Đọc tử số/mẫu số, coverage và status trước tỷ lệ. not_in_release không có nghĩa không có lỗi; thiếu audit hiện công khai.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Checksum đúng không chứng minh DQ tốt. Không tải raw hoặc tính lại audit để lấp chỗ trống.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 7:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 7
summary_views.append(render_summary_section(release, 2))

_pubg_progress['12_final_results_summary.ipynb'] = 9


### 3. Kết quả RQ1

Câu hỏi: Hành vi liên hệ survival/placement trong mode và scope nào?

Input/phương pháp: Đọc Pearson/Spearman/N/status/primary validity đã tính. Pearson đo liên hệ tuyến tính; Spearman đo liên hệ đơn điệu theo thứ hạng.

Cách đọc: Giữ dấu hệ số; so cùng target/mode/N. NA do constant/insufficient không là hệ số 0; p nhỏ không thay effect size.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Target-derived và phase-survival coupling chỉ diagnostic; association không nhân quả, chưa có CI nếu producer chưa tính.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 9:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 9
summary_views.append(render_summary_section(release, 3))

_pubg_progress['12_final_results_summary.ipynb'] = 11


### 4. Kết quả RQ2

Câu hỏi: Những profile hành vi và độ ổn định C1-C5 được quan sát ra sao?

Input/phương pháp: Đọc centers/sizes/retention/K/robustness/C5 của KMeans đã khóa. Standardized center là theo scaler fit; outcome chỉ dùng post-hoc sau assignment.

Cách đọc: Cluster ID cục bộ trong từng mode; không so cluster0 Solo với cluster0 Duo. ARI đo mức nhất quán assignments, không đo chiến thuật tốt.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Duration proxy, representation, K và minimum games ảnh hưởng profile; diagnostics sample không là full clustering hoặc heldout generalization.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 11:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 11
summary_views.append(render_summary_section(release, 4))

_pubg_progress['12_final_results_summary.ipynb'] = 13


### 5. RQ3: thời gian sống sót

Câu hỏi: S1/S2 so với train mean/median đạt các số đo nào?

Input/phương pháp: Đọc micro/match-aware metric và feature/cohort records. MAE/RMSE là sai số cùng đơn vị target; R2 không phải accuracy.

Cách đọc: So cùng population/split/recipe. Đơn vị time pending không tự gọi giây; R2 undefined có reason, không gán 0.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; S1 hồi cứu sau trận; S2 chỉ future claim theo chronology/availability đã khóa. Grade C phải hiện blocked, không có score giả.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 13:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 13
summary_views.append(render_summary_section(release, 5))

_pubg_progress['12_final_results_summary.ipynb'] = 15


### 6. RQ3: thứ hạng chuẩn hóa

Câu hỏi: P1/P2/P3 và baselines được đánh giá trong scope nào?

Input/phương pháp: Normalized placement trong [0,1], 1 là đầu, 0 là cuối; target đội lặp trên player rows. Đọc micro/match/team-aware và N riêng.

Cách đọc: P1 có survival trực tiếp; P2 bỏ cột đó nhưng có thể còn survival-derived descendants theo D02. Không gọi P2 độc lập mọi thông tin survival.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; P1/P2 là hồi cứu, không dự đoán trước trận. P3 conditional Grade A/B; team dependence không biến mất khi N player lớn.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 15:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 15
summary_views.append(render_summary_section(release, 6))

_pubg_progress['12_final_results_summary.ipynb'] = 17


### 7. Đóng góp Combat Timing T0/T1

Câu hỏi: Thêm timing thay đổi sai số trên cùng cohort thế nào?

Input/phương pháp: Đọc paired comparisons T1-T0, same rows/split/model/preprocessing và coverage đã khóa. Không bootstrap lại hoặc chọn estimator từ test.

Cách đọc: Delta MAE/RMSE âm nghĩa candidate sai số thấp hơn; delta R2 dương khác ý nghĩa. CI chứa0 chưa chứng minh chiều cải thiện.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Placement neo P2; survival nếu đăng ký chỉ timing subset an toàn D01. Missing event có thể đổi population, cần đọc coverage.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 17:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 17
summary_views.append(render_summary_section(release, 7))

_pubg_progress['12_final_results_summary.ipynb'] = 19


### 8. Loại nhóm đặc trưng và importance

Câu hỏi: Bỏ Combat/Movement/Support/Timing thay đổi kết quả ra sao?

Input/phương pháp: Đọc ablation và importance đã khóa. Removal gồm raw/derived descendants/indicators; không chạy selection mới. Native/permutation importance có ý nghĩa khác nhau.

Cách đọc: Candidate là nhánh đã bỏ nhóm, reference là full; delta và CI theo cùng recipe. N và feature list cho thấy removal thực tế.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Importance/correlation không chứng minh causal. Collinearity khiến importance chia sẻ giữa biến; group ablation là evidence nhóm.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 19:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 19
summary_views.append(render_summary_section(release, 8))

_pubg_progress['12_final_results_summary.ipynb'] = 21


### 9. Phân tích sai số

Câu hỏi: Mode/history depth/target regions nào có sai số và coverage đáng chú ý?

Input/phương pháp: Đọc slices/residual summaries/bins đã đăng ký trước test. Residual=y-prediction; MAE/RMSE và signed residual không đồng nghĩa.

Cách đọc: Đọc n_observations/n_matches/coverage/status/unit; low-N/empty/outside bins phải hiện reason, không xếp hạng chắc chắn.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Không chọn lại bins từ test hoặc suy metric thiếu. Slice population khác nhau không phải so sánh nhân quả giữa mode.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 21:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 21
summary_views.append(render_summary_section(release, 9))

_pubg_progress['12_final_results_summary.ipynb'] = 23


### 10. Độ bất định

Câu hỏi: CI được tính bằng resampling unit, seed và budget nào?

Input/phương pháp: Đọc saved comparisons/bootstrap receipts và hình đã khóa. Match paired bootstrap dùng cùng match indices có lặp cho hai nhánh; CI percentile95%.

Cách đọc: Đối chiếu confidence, valid replicates, seed, N và reason. N mô phỏng bootstrap không phải N độc lập tăng thêm.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; CI thiếu giữ null/reason, không tự vẽ mới. Match bootstrap chưa giải quyết hoàn toàn player lặp xuyên trận; time/compute budgets chưa là peak proof.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 23:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 23
summary_views.append(render_summary_section(release, 10))

_pubg_progress['12_final_results_summary.ipynb'] = 25


### 11. Phát hiện chính có truy nguồn

Câu hỏi: Mỗi phát biểu có số đo, scope, N, run và bảng nguồn nào?

Input/phương pháp: Chép giá trị từ summary rows đã khóa thành phát biểu trung tính, không tính metric mới hoặc chọn overall winner. Findings đọc mọi summary chunk, không chọn từ preview100dòng.

Cách đọc: Mỗi dòng gồm số đo, CI hoặc cảnh báo thiếu, mode/population, run, SHA và source table; mở bảng nguồn trước khi đưa vào báo cáo.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Fixture findings không là kết quả PUBG. Không dùng từ gây ra/giúp thắng; effect không đồng nghĩa mức độ tác động nhân quả.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 25:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 25
summary_views.append(render_summary_section(release, 11))

_pubg_progress['12_final_results_summary.ipynb'] = 27


### 12. Hạn chế và ghi chú bàn giao

Câu hỏi: Những exclusions, giới hạn và quyết định còn mở nào phải giữ khi viết báo cáo?

Input/phương pháp: Đọc required matrix locked: blocked/resource_limited/failed/reason, hình diagnostic và prior test exposure. Không dùng trạng thái notebook latest để thay lịch sử release.

Cách đọc: Không giấu S2/P3 Grade C hoặc optional failures. unknown exposure không đồng nghĩa chưa từng xem test; report_ready không phải chứng nhận mọi kết luận đúng.

Kết luận và giới hạn: chỉ dùng số đo/source hiện ngay dưới; Dữ liệu quan sát/hồi cứu; identity name không bất biến; event mismatch, proxy duration, chronology granularity/ties, repeated teams/players, K sensitivity, dataset age và unobserved gameplay. Colab/GPU/Drive/full-data thật chưa chứng minh từ fixture.

Kiểm soát chất lượng: thiếu selected artifact hiện not_in_release/pending hoặc status/reason; không suy diễn số hay đọc file ngoài manifest.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 27:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 27
summary_views.append(render_summary_section(release, 12))

_pubg_progress['12_final_results_summary.ipynb'] = 29


### Bàn giao phiên bản đã đọc

Bảng dưới nối relative path, persistent path, checksum và release ID để thành viên tiếp theo mở đúng bản. Các file thuộc snapshot không bị ghi đè; canonical chỉ là lối vào đã chọn. Notebook không ghi handover/checkpoint mới. Sau khi lưu notebook có output bằng thao tác riêng, không gọi việc đó là chạy lại pipeline.

Tái huấn luyện vẫn cần raw/processed, môi trường và approvals tương thích; đọc release không cần chúng. Dừng tại báo cáo này; các kiểm tra G0/toàn pipeline và môi trường thật thuộc giai đoạn15/16, không tự chứng nhận từ summary.


In [ ]:
if "manifest_path" not in globals() or "PROJECT_ROOT" not in globals():
    raise RuntimeError("Runtime đã mất trạng thái. Chạy lại cell Chọn nơi lưu dữ liệu và Bootstrap, rồi cell khởi tạo stage trước khi tiếp tục.")
_pubg_progress = globals().setdefault("_PUBG_CELL_PROGRESS", {})
if _pubg_progress.get('12_final_results_summary.ipynb', -1) < 29:
    raise RuntimeError("12_final_results_summary.ipynb: Chạy thành công cell trước trước khi tiếp tục; không bỏ qua cell bị lỗi.")
_pubg_progress['12_final_results_summary.ipynb'] = 29
display(pd.DataFrame([{**row, 'Persistent path': str(release['root']/row['path']),
    'Release': release['manifest']['release_id'], 'Consumer': 'Báo cáo nhóm, không chạy tiếp thí nghiệm'} for row in release['audit']]))
display(pd.DataFrame(summary_views))

_pubg_progress['12_final_results_summary.ipynb'] = 31
